In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.db import read_sql_file

In [2]:
metrics = read_sql_file("09_metrics_by_arm.sql")
# Transpose so that each metric is a row and each arm is a column, which is easier to read
metrics.set_index("arm").T

arm,control,mens,womens
n_customers,21306.00000,21307.00000,21387.00000
visit_rate,0.10617,0.18276,0.15140
visit_rate_diff,0.00000,0.07659,0.04523
visit_rate_lift,0.00000,0.72140,0.42610
conversion_rate,0.00573,0.01253,0.00884
conversion_rate_diff,0.00000,0.00681,0.00311
conversion_rate_lift,0.00000,1.18840,0.54330
spend_per_customer,0.65280,1.42260,1.07720
spend_per_customer_diff,0.00000,0.76980,0.42440
spend_per_customer_lift,0.00000,1.17930,0.65020


In [3]:
summary = read_sql_file("10_arm_summary_stats.sql").set_index("arm")
# SUM() over integers arrives as Decimal; convert to plain numeric types
summary = summary.astype(
    {"n": int, "visits": int, "conversions": int, "spend_mean": float, "spend_var": float}
)
summary

,n,visits,conversions,spend_mean,spend_var
arm,,,,,
control,21306,2262,122,0.652789,134.286372
mens,21307,3894,267,1.422617,315.211799
womens,21387,3238,189,1.077202,228.496675


In [4]:
from src.stats import diff_in_proportions, diff_in_means_welch

t = summary.loc["mens"]
c = summary.loc["control"]

mens_vs_control = pd.DataFrame({
    "visit": diff_in_proportions(t["visits"], t["n"], c["visits"], c["n"]),
    "conversion": diff_in_proportions(t["conversions"], t["n"], c["conversions"], c["n"]),
    "spend": diff_in_means_welch(
        t["spend_mean"], t["spend_var"], t["n"],
        c["spend_mean"], c["spend_var"], c["n"],
    ),
}).T

mens_vs_control

,treatment_value,control_value,diff,ci_low,ci_high,relative_lift,test_statistic,p_value
visit,0.182757,0.106167,0.076590,0.069954,0.083226,0.721405,22.486041,5.685165e-112
conversion,0.012531,0.005726,0.006805,0.005000,0.008610,1.188422,7.385114,1.523224e-13
spend,1.422617,0.652789,0.769827,0.485140,1.054515,1.179289,5.300140,1.163815e-07


In [5]:
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

# Conversion: statsmodels uses the pooled standard error by default, same as our function
z, p = proportions_ztest(
    count=[t["conversions"], c["conversions"]], nobs=[t["n"], c["n"]]
)
print(f"statsmodels conversion z-test: z = {z:.4f}, p = {p:.3e}")

# Spend: scipy's Welch t-test computed from summary statistics
res = stats.ttest_ind_from_stats(
    t["spend_mean"], np.sqrt(t["spend_var"]), t["n"],
    c["spend_mean"], np.sqrt(c["spend_var"]), c["n"],
    equal_var=False,
)
print(f"scipy Welch t-test on spend:   t = {res.statistic:.4f}, p = {res.pvalue:.3e}")

statsmodels conversion z-test: z = 7.3851, p = 1.523e-13
scipy Welch t-test on spend:   t = 5.3001, p = 1.164e-07


In [8]:
from statsmodels.stats.multitest import multipletests
from src.stats import compare_arms

comparisons = [("mens", "control"), ("womens", "control"), ("mens", "womens")]
results = pd.concat(
    [compare_arms(summary, t, c) for t, c in comparisons], ignore_index=True
)

# Holm correction across the whole family of 9 tests (controls the family-wise error rate)
reject, p_holm, _, _ = multipletests(results["p_value"], alpha=0.05, method="holm")
results["p_value_holm"] = p_holm
results["significant_holm"] = reject

units = {"visit": "proportion", "conversion": "proportion", "spend": "USD per customer"}
results.insert(2, "unit", results["metric"].map(units))

In [11]:
display_cols = [
    "comparison", "metric", "unit", "diff", "ci_low", "ci_high",
    "relative_lift", "p_value", "p_value_holm", "significant_holm",
]
results[display_cols].style.format({
    "diff": "{:.4f}",
    "ci_low": "{:.4f}",
    "ci_high": "{:.4f}",
    "relative_lift": "{:.1%}",
    "p_value": "{:.2e}",
    "p_value_holm": "{:.2e}",
})

,comparison,metric,unit,diff,ci_low,ci_high,relative_lift,p_value,p_value_holm,significant_holm
0,mens vs control,visit,proportion,0.0766,0.0700,0.0832,72.1%,5.69e-112,5.12e-111,True
1,mens vs control,conversion,proportion,0.0068,0.0050,0.0086,118.8%,1.52e-13,9.14e-13,True
2,mens vs control,spend,USD per customer,0.7698,0.4851,1.0545,117.9%,1.16e-07,5.82e-07,True
3,womens vs control,visit,proportion,0.0452,0.0389,0.0516,42.6%,3.18e-44,2.55e-43,True
4,womens vs control,conversion,proportion,0.0031,0.0015,0.0047,54.3%,1.57e-04,6.28e-04,True
5,womens vs control,spend,USD per customer,0.4244,0.1690,0.6799,65.0%,1.13e-03,2.26e-03,True
6,mens vs womens,visit,proportion,0.0314,0.0243,0.0384,20.7%,3.80e-18,2.66e-17,True
7,mens vs womens,conversion,proportion,0.0037,0.0017,0.0056,41.8%,2.05e-04,6.28e-04,True
8,mens vs womens,spend,USD per customer,0.3454,0.0326,0.6583,32.1%,3.05e-02,3.05e-02,True


## Results: treatment effects

Differences are treatment minus comparison arm, with unadjusted 95% confidence intervals.
p-values are Holm-adjusted across all 9 tests. Spend is measured over the 2-week window after the campaign.
The primary metric is spend per customer; visit and conversion rates are secondary.

| Comparison | Visit rate | Conversion rate | Spend per customer |
|---|---|---|---|
| Mens vs control | +7.66 pp [7.00, 8.32] | +0.68 pp [0.50, 0.86] | +$0.77 [0.49, 1.05] |
| Womens vs control | +4.52 pp [3.89, 5.16] | +0.31 pp [0.15, 0.47] | +$0.42 [0.17, 0.68] |
| Mens vs womens | +3.14 pp [2.43, 3.84] | +0.37 pp [0.17, 0.56] | +$0.35 [0.03, 0.66] |

- Both emails significantly increase visits, conversions and spend compared with no email.
- The mens email outperforms the womens email on all three metrics.
- The evidence that mens beats womens on spend is the weakest (Holm-adjusted p = 0.031). It would not
  survive a Bonferroni correction (p = 0.27), and the lower bound of its confidence interval is close to zero.